In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

PROC = Path("../data/processed")

In [2]:
listings  = pd.read_csv(PROC / "listings_clean.csv", low_memory=False)
cal_list  = pd.read_csv(PROC / "calendar_listing_summary.csv")
cal_daily = pd.read_csv(PROC / "calendar_daily_city.csv", parse_dates=["date"])
weather   = pd.read_csv(PROC / "weather_clean.csv", parse_dates=["date", "calendar_date"])
economy   = pd.read_csv(PROC / "economy_clean.csv")
landmarks = pd.read_csv(PROC / "landmarks_clean.csv")

for name, df in [("listings", listings), ("cal_list", cal_list), ("cal_daily", cal_daily),
                 ("weather", weather), ("economy", economy), ("landmarks", landmarks)]:
    print(f"{name:10s} {df.shape}")

listings   (41183, 71)
cal_list   (41183, 7)
cal_daily  (1463, 5)
weather    (1472, 7)
economy    (32, 6)
landmarks  (2443, 6)


In [3]:
print(listings[["city", "id"]].dtypes)
print(cal_list[["city", "listing_id"]].dtypes)

print("\nlistings rows:", len(listings), "| unique (city,id):", listings[["city", "id"]].drop_duplicates().shape[0])
print("cal_list rows:", len(cal_list), "| unique (city,listing_id):", cal_list[["city", "listing_id"]].drop_duplicates().shape[0])

m = listings[["city", "id"]].merge(cal_list[["city", "listing_id"]],
        left_on=["city", "id"], right_on=["city", "listing_id"], how="left", indicator=True)
print("\nlistings without calendar summary:", (m["_merge"] == "left_only").sum())

city    object
id       int64
dtype: object
city          object
listing_id     int64
dtype: object

listings rows: 41183 | unique (city,id): 41183
cal_list rows: 41183 | unique (city,listing_id): 41183

listings without calendar summary: 0


In [4]:
cal_cols = cal_list.rename(columns={
    "n_days": "cal_n_days",
    "days_available": "cal_days_available",
    "share_available": "cal_share_available",
    "min_nights_min": "cal_min_nights_min",
    "min_nights_max": "cal_min_nights_max",
})

lf = listings.merge(
    cal_cols,
    left_on=["city", "id"], right_on=["city", "listing_id"],
    how="left", validate="one_to_one",
).drop(columns=["listing_id"])

print("rows:", len(lf), "(41183)")
print("columns:", lf.shape[1])
print("missing share_available:", lf["cal_share_available"].isna().sum())

rows: 41183 (41183)
columns: 76
missing share_available: 0


In [5]:
print(lf.groupby("city")[["cal_share_available", "availability_365"]].mean().round(3))
print(lf[["cal_share_available", "availability_365"]].corr().round(3))

           cal_share_available  availability_365
city                                            
Amsterdam                0.276            93.621
Barcelona                0.581           180.479
Berlin                   0.442           158.328
Columbus                 0.656           239.105
                     cal_share_available  availability_365
cal_share_available                1.000             0.898
availability_365                   0.898             1.000


In [6]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = p2 - p1
    dlmb = np.radians(lon2) - np.radians(lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlmb / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

CENTER = {"Amsterdam": (52.3676, 4.9041), "Barcelona": (41.3874, 2.1686),
          "Berlin": (52.5200, 13.4050), "Columbus": (39.9612, -82.9988)}

In [7]:
parts = []
for city, grp in lf.groupby("city"):
    lm = landmarks[landmarks["city"] == city]
    ok = grp["latitude"].notna() & grp["longitude"].notna()
    g = grp.loc[ok]

    D = haversine_km(g["latitude"].to_numpy()[:, None], g["longitude"].to_numpy()[:, None],
                     lm["latitude"].to_numpy()[None, :], lm["longitude"].to_numpy()[None, :])

    out = pd.DataFrame({
        "dist_nearest_landmark_km": D.min(axis=1),
        "landmarks_1km": (D <= 1).sum(axis=1),
        "landmarks_2km": (D <= 2).sum(axis=1),
        "dist_center_km": haversine_km(g["latitude"].to_numpy(), g["longitude"].to_numpy(),
                                       CENTER[city][0], CENTER[city][1]),
    }, index=g.index)
    parts.append(out)
    print(city, "listings:", len(g), "| landmarks:", len(lm))

geo = pd.concat(parts)
lf = lf.join(geo)
print("\ncolumns:", lf.shape[1], "| rows:", len(lf))

Amsterdam listings: 10612 | landmarks: 558
Barcelona listings: 15236 | landmarks: 508
Berlin listings: 12376 | landmarks: 912
Columbus listings: 2959 | landmarks: 465

columns: 80 | rows: 41183


In [8]:
cols = ["dist_nearest_landmark_km", "landmarks_1km", "landmarks_2km", "dist_center_km"]
print(lf[cols].isna().sum())
print(lf.groupby("city")[cols].median().round(2))

dist_nearest_landmark_km    0
landmarks_1km               0
landmarks_2km               0
dist_center_km              0
dtype: int64
           dist_nearest_landmark_km  landmarks_1km  landmarks_2km  \
city                                                                
Amsterdam                      0.21           17.0           97.0   
Barcelona                      0.17           29.0          139.0   
Berlin                         0.32            8.0           40.0   
Columbus                       0.35            9.0           48.0   

           dist_center_km  
city                       
Amsterdam            2.56  
Barcelona            1.58  
Berlin               4.23  
Columbus             3.48  


In [9]:
print(landmarks.groupby("city")["distance_from_center_m"].describe().round(0))

           count    mean     std    min     25%     50%     75%     max
city                                                                   
Amsterdam  558.0  2935.0  2450.0   73.0   992.0  1969.0  4152.0  9766.0
Barcelona  508.0  3314.0  2464.0  142.0  1321.0  2563.0  4751.0  9962.0
Berlin     912.0  3674.0  2719.0    0.0  1525.0  2768.0  5557.0  9980.0
Columbus   465.0  2567.0  2474.0   22.0   639.0  1708.0  4065.0  9984.0


In [10]:
eco = (economy[economy["year"] == 2024]
       [["city", "gdp_per_capita_usd", "inflation_pct"]]
       .rename(columns={"gdp_per_capita_usd": "country_gdp_per_capita_usd",
                        "inflation_pct": "country_inflation_pct"}))

lf = lf.merge(eco, on="city", how="left", validate="many_to_one")
print("rows:", len(lf), "| columns:", lf.shape[1])
print(lf[["country_gdp_per_capita_usd", "country_inflation_pct"]].isna().sum())
print(lf.groupby("city")[["country_gdp_per_capita_usd", "country_inflation_pct"]].first().round(2))

rows: 41183 | columns: 82
country_gdp_per_capita_usd    0
country_inflation_pct         0
dtype: int64
           country_gdp_per_capita_usd  country_inflation_pct
city                                                        
Amsterdam                    67465.32                   3.35
Barcelona                    35326.77                   2.77
Berlin                       56103.73                   2.26
Columbus                     86169.66                   2.95


In [12]:
cd = cal_daily.merge(
    weather[["city", "calendar_date", "temp_max", "temp_min", "temp_mean", "precipitation"]],
    left_on=["city", "date"], right_on=["city", "calendar_date"],
    how="left", validate="one_to_one",
).drop(columns=["calendar_date"])

print("rows:", len(cd), "(1463)")
print(cd[["temp_max", "temp_min", "temp_mean", "precipitation"]].isna().sum())

rows: 1463 (1463)
temp_max         5
temp_min         5
temp_mean        5
precipitation    5
dtype: int64


In [13]:
cd["month"] = cd["date"].dt.month
cd["day_of_week"] = cd["date"].dt.dayofweek
cd["is_weekend"] = cd["day_of_week"].isin([4, 5])
cd["in_common_period"] = (cd["date"] >= "2026-09-25") & (cd["date"] <= "2027-09-16")

print(cd.groupby("city")["in_common_period"].sum())
print(cd.head())

city
Amsterdam    357
Barcelona    357
Berlin       357
Columbus     357
Name: in_common_period, dtype: int64
        city       date  n_listings  n_available  share_available  temp_max  \
0  Amsterdam 2026-09-17       10612        755.0           0.0711      17.4   
1  Amsterdam 2026-09-18       10612        990.0           0.0933      20.5   
2  Amsterdam 2026-09-19       10612       1187.0           0.1119      24.1   
3  Amsterdam 2026-09-20       10612       1915.0           0.1805      22.2   
4  Amsterdam 2026-09-21       10612       1936.0           0.1824      19.4   

   temp_min  temp_mean  precipitation  month  day_of_week  is_weekend  \
0      13.1      15.25            9.7      9            3       False   
1      17.0      18.75            5.5      9            4        True   
2      14.9      19.50            0.0      9            5        True   
3      17.2      19.70            1.2      9            6       False   
4      11.4      15.40            0.2      9      

In [14]:
miss = cd[cd["temp_max"].isna()]
print(miss[["city", "date", "in_common_period"]])

          city       date  in_common_period
1458  Columbus 2027-09-20             False
1459  Columbus 2027-09-21             False
1460  Columbus 2027-09-22             False
1461  Columbus 2027-09-23             False
1462  Columbus 2027-09-24             False


In [15]:
lf.to_parquet(PROC / "listings_final.parquet", index=False)
lf.to_csv(PROC / "listings_final.csv", index=False)
cd.to_parquet(PROC / "city_daily_final.parquet", index=False)
cd.to_csv(PROC / "city_daily_final.csv", index=False)
print("saved:", sorted(p.name for p in PROC.iterdir()))

saved: ['calendar_daily_city.csv', 'calendar_listing_summary.csv', 'city_daily_final.csv', 'city_daily_final.parquet', 'economy_clean.csv', 'economy_clean.parquet', 'landmarks_clean.csv', 'landmarks_clean.parquet', 'listings_clean.csv', 'listings_final.csv', 'listings_final.parquet', 'weather_clean.csv', 'weather_clean.parquet']


In [16]:
chk = pd.read_parquet(PROC / "listings_final.parquet")
print("listings_final:", chk.shape, "| unique (city,id):", chk[["city", "id"]].drop_duplicates().shape[0])
print("price available:", chk["price"].notna().sum(), "| price_missing flag:", chk["price_missing"].sum())

chk2 = pd.read_parquet(PROC / "city_daily_final.parquet")
print("city_daily_final:", chk2.shape)
print(chk2.groupby("city").size())

listings_final: (41183, 82) | unique (city,id): 41183
price available: 29440 | price_missing flag: 11743
city_daily_final: (1463, 13)
city
Amsterdam    365
Barcelona    367
Berlin       366
Columbus     365
dtype: int64
